In [51]:
from t2m import Text2Motion
from utils.get_opt import get_opt
from utils.fixseed import fixseed

import torch
import numpy as np
from os.path import join as pjoin

In [52]:
denoiser_name = "denoiser_example"
dataset_name = "t2m"
generator = Text2Motion(denoiser_name, dataset_name)

opt = generator.opt
wrapper_opt = get_opt(opt.dataset_opt_path, torch.device("cuda"))
mean = np.load(pjoin(wrapper_opt.meta_dir, "mean.npy"))
std = np.load(pjoin(wrapper_opt.meta_dir, "std.npy"))

Reading checkpoints/t2m/denoiser_example/opt.txt
Reading checkpoints/t2m/vae_example/opt.txt
Loading VAE Model vae_example
Loading Denoiser Model denoiser_example


0it [00:00, ?it/s]

Loaded CLIP text encoder version ViT-B/32
Reading ./checkpoints/t2m/Comp_v6_KLD005/opt.txt


### Original

In [53]:
fixseed(42)
src_text = "a person stands still and kicks their right leg forward"
m_lens = 120
cfg_scale = 7.5
num_inference_timesteps = 200

init_noise, src_motion, (sa, ta, ca) = generator.generate(src_text,
                                                          m_lens,
                                                          cfg_scale,
                                                          num_inference_timesteps)

### Edit - 4 Different Cases

In [54]:
# edit_text = "slowly"
#edit_text = "a man is walking while waving his right hand"
#src_proportion = 0.2

# # case 1: mirror
# edit_motion = generator.edit(init_noise,
#                              src_text=src_text,
#                              edit_text=edit_text,
#                              edit_mode="mirror",
#                              mirror_mode="lower",
#                              cfg_scale=cfg_scale,
#                              num_inference_timesteps=num_inference_timesteps,
#                              src_sa=sa,
#                              src_ta=ta,
#                              src_ca=ca,
#                              src_proportion=src_proportion)

# # case 2: reweight
# edit_motion = generator.edit(init_noise,
#                              src_text=src_text,
#                              edit_text=src_text,
#                              edit_mode="reweight",
#                              tgt_word="high",
#                              reweight_scale=-1.0,
#                              cfg_scale=cfg_scale,
#                              num_inference_timesteps=num_inference_timesteps,
#                              src_sa=sa,
#                              src_ta=ta,
#                              src_ca=ca,
#                              src_proportion=src_proportion)

# case 3: refine
#edit_motion = generator.edit(init_noise,
                             #src_text=src_text,
                             #edit_text=edit_text,
                             #edit_mode="refine",
                             #cfg_scale=cfg_scale,
                             #num_inference_timesteps=num_inference_timesteps,
                             #src_sa=sa,
                             #src_ta=ta,
                             #src_ca=ca,
                             #src_proportion=src_proportion)

# # case 4: word swap
# edit_motion = generator.edit(init_noise,
#                              src_text=src_text,
#                              edit_text=edit_text,
#                              edit_mode="word_swap",
#                              cfg_scale=cfg_scale,
#                              num_inference_timesteps=num_inference_timesteps,
#                              src_sa=sa,
#                              src_ta=None,
#                              src_ca=ca,
#                              src_proportion=src_proportion,
#                              swap_src_proportion=0.2)


### Visualize

In [55]:
import os
from os.path import join as pjoin
import torch
import numpy as np
from utils.motion_process import recover_from_ric
from utils.plot_script import plot_3d_motion
from utils.get_opt import get_opt

def plot_t2m(data, text, filename):
    os.makedirs("edit_result", exist_ok=True)
    #data = data[:m_lens[0].item()]
    data = data[:m_lens]
    joint = recover_from_ric(torch.from_numpy(data).float(), opt.joints_num).numpy()
    save_path = pjoin("edit_result", f"{filename}.mp4")
    plot_3d_motion(save_path, opt.kinematic_chain, joint, title=text, fps=20)

    np.save(pjoin("edit_result", f"{filename}_pos.npy"), joint)
    np.save(pjoin("edit_result", f"{filename}_feats.npy"), data)
    
# mean and std for de-normalization
wrapper_opt = get_opt(opt.dataset_opt_path, torch.device('cuda'))
mean = np.load(pjoin(wrapper_opt.meta_dir, 'mean.npy'))
std = np.load(pjoin(wrapper_opt.meta_dir, 'std.npy'))

Reading ./checkpoints/t2m/Comp_v6_KLD005/opt.txt


### Plot Motions

In [56]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

src_motion = src_motion.cpu().numpy() * std + mean
plot_t2m(src_motion[0], src_text, "src")

#edit_motion = edit_motion.detach().cpu().numpy() * std + mean
#plot_t2m(edit_motion[0], edit_text, "edit")

### Video Visualization

In [57]:
#from moviepy.editor import VideoFileClip, clips_array
#src_video = VideoFileClip(pjoin("edit_result", "src.mp4"))
#edit_video = VideoFileClip(pjoin("edit_result", "edit.mp4"))
#final_video = clips_array([[src_video, edit_video]])
#final_video.write_videofile(pjoin("edit_result", "final.mp4"))


from IPython.display import Video
Video("edit_result/src.mp4", width=800, height=400,embed=True)

In [58]:
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.axes3d as p3
import numpy as np

joint = np.load("edit_result/src_pos.npy")
fig = plt.figure()
ax = p3.Axes3D(fig)
ax.plot3D(joint[0, :, 0], joint[0, :, 1], joint[0, :, 2], 'ro')
plt.show()

/tmp/ipykernel_111851/2479854159.py:9: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [59]:
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.axes3d as p3
import numpy as np

joint = np.load("edit_result/src_pos.npy")
fig = plt.figure()
ax = p3.Axes3D(fig)
ax.plot3D(joint[0, :, 0], joint[0, :, 1], joint[0, :, 2], 'ro')
plt.savefig("edit_result/test_frame.png")
print("保存成功")

保存成功


In [60]:
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d.axes3d as p3
import numpy as np

joint = np.load("edit_result/src_pos.npy")
fig = plt.figure()
ax = p3.Axes3D(fig)

def update(index):
    ax.cla()
    ax.plot3D(joint[index, :, 0], joint[index, :, 1], joint[index, :, 2], 'ro')

ani = FuncAnimation(fig, update, frames=len(joint), interval=50)
ani.save("edit_result/src.gif", writer='pillow', fps=20)
print("gif保存成功")

gif保存成功
